In [2]:
import os
import sys

sys.path.append(os.path.abspath('..'))

import torch
from src.losses import ComboLoss

criterion = ComboLoss()

# Dummy batch of 2 images, 4 channels, 128x800
dummy_logits = torch.randn(2, 4, 128, 800, requires_grad=True)
dummy_targets = torch.zeros(2, 4, 128, 800)  # clean images test
dummy_targets[0, 2, 20:50, 100:200] = 1.0  # add one defect

loss = criterion(dummy_logits, dummy_targets)
loss.backward()  # Test that gradients compute without error

print(f"Computed Loss: {loss.item():.4f}")
print("Gradients successfully calculated!")


Computed Loss: 1.7979
Gradients successfully calculated!


In [3]:
import os
import sys

sys.path.append(os.path.abspath('..'))

import numpy as np
from src.metrics import compute_competition_metrics, post_process_mask
import torch

print("=== 1. Testing post_process_mask ===")
# Create a dummy mask for 4 classes: shape (4, 128, 800)
sample_mask = np.zeros((4, 128, 800), dtype=np.uint8)

# Class 0 (defect_a, threshold=15):
# Add a blob of area 20 (>=15, should KEEP) and a tiny speckle of area 5 (<15, should REMOVE)
sample_mask[0, 10:15, 10:14] = 1  # 5x4 = 20 pixels
sample_mask[0, 50:51, 50:55] = 1  # 1x5 = 5 pixels

# Class 1 (defect_b, threshold=50):
# Add a blob of area 60 (>=50, should KEEP) and a blob of area 30 (<50, should REMOVE)
sample_mask[1, 20:26, 20:30] = 1  # 6x10 = 60 pixels
sample_mask[1, 70:73, 70:80] = 1  # 3x10 = 30 pixels

clean_mask = post_process_mask(sample_mask)

print(
    f"Class 0 (defect_a): Original = {sample_mask[0].sum()} px -> Cleaned ="
    f" {clean_mask[0].sum()} px (Expected: 20)"
)
print(
    f"Class 1 (defect_b): Original = {sample_mask[1].sum()} px -> Cleaned ="
    f" {clean_mask[1].sum()} px (Expected: 60)"
)

assert clean_mask[0].sum() == 20, "Class 0 filter error!"
assert clean_mask[1].sum() == 60, "Class 1 filter error!"
print("✅ post_process_mask passed all tests!\n")


print("=== 2. Testing compute_competition_metrics with Torch Tensors ===")
# Create dummy predictions and ground truth: (Batch=2, Channels=4, 128, 800)
dummy_logits = torch.randn(2, 4, 128, 800)
dummy_targets = torch.zeros(2, 4, 128, 800)
# Add a real defect to image 0, channel 2 (defect_c)
dummy_targets[0, 2, 30:70, 200:300] = 1.0

# Compute competition metrics
results = compute_competition_metrics(
    dummy_logits, dummy_targets, threshold=0.5
)

for k, v in results.items():
  print(f"{k}: {v:.4f}")

print("✅ compute_competition_metrics passed without error!")


=== 1. Testing post_process_mask ===
Class 0 (defect_a): Original = 25 px -> Cleaned = 20 px (Expected: 20)
Class 1 (defect_b): Original = 90 px -> Cleaned = 60 px (Expected: 60)
✅ post_process_mask passed all tests!

=== 2. Testing compute_competition_metrics with Torch Tensors ===
foreground_dice: 0.0719
false_positive_score: 0.4991
empty_accuracy: 0.0000
official_score: 0.1501
✅ compute_competition_metrics passed without error!
